In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. ЗАГРУЗКА 8 ТАБЛИЦ
# ==========================================

DATA_PATH = Path(r"C:\Users\User\Documents\DA-businesskeys-dunnhanby-main\data\processed")

tables = {}

for file in DATA_PATH.glob("*"):
    if file.suffix.lower() == ".csv":
        tables[file.stem] = pd.read_csv(file)

print(f"Количество загруженных таблиц: {len(tables)}")
print("\nНазвания таблиц:")

for name, df in tables.items():
    print(f"{name}: {df.shape}")


# ==========================================
# 2. ОБЩАЯ СТРУКТУРА ДАННЫХ
# ==========================================

overview = []

for name, df in tables.items():
    overview.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / 1024**2, 2
        ),
        "duplicates": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "missing_percentage": round(
            df.isna().mean().mean() * 100, 2
        )
    })

overview_df = pd.DataFrame(overview)

display(overview_df)


# ==========================================
# 3. ДЕТАЛЬНОЕ ИССЛЕДОВАНИЕ КАЖДОЙ ТАБЛИЦЫ
# ==========================================

for name, df in tables.items():

    print("\n" + "=" * 70)
    print(f"TABLE: {name}")
    print("=" * 70)

    # Первые строки
    print("\n1. FIRST 5 ROWS")
    display(df.head())

    # Последние строки
    print("\n2. LAST 5 ROWS")
    display(df.tail())

    # Типы данных
    print("\n3. DATA TYPES")
    display(df.dtypes.to_frame("dtype"))

    # Пропуски
    print("\n4. MISSING VALUES")

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2)
    })

    display(missing[missing["missing_count"] > 0])

    # Дубликаты
    print("\n5. DUPLICATES")
    print("Exact duplicate rows:", df.duplicated().sum())

    # Уникальные значения
    print("\n6. UNIQUE VALUES")
    unique = df.nunique().to_frame("unique_values")
    display(unique)

    # Статистика
    print("\n7. DESCRIPTIVE STATISTICS")
    display(df.describe(include="all").T)


# ==========================================
# 4. ПРОФИЛЬ СТОЛБЦОВ ВСЕХ ТАБЛИЦ
# ==========================================

column_profiles = []

for name, df in tables.items():

    for col in df.columns:

        column_profiles.append({
            "table": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": df[col].count(),
            "missing": df[col].isna().sum(),
            "missing_pct": round(
                df[col].isna().mean() * 100, 2
            ),
            "unique": df[col].nunique(),
            "unique_pct": round(
                df[col].nunique() / max(len(df), 1) * 100, 2
            )
        })

column_profiles_df = pd.DataFrame(column_profiles)

display(column_profiles_df)


# ==========================================
# 5. ПОИСК ПОТЕНЦИАЛЬНЫХ КЛЮЧЕЙ
# ==========================================

print("\nPOTENTIAL KEY COLUMNS")

for name, df in tables.items():

    print(f"\n{name}")

    for col in df.columns:

        unique_count = df[col].nunique()
        non_null_count = df[col].count()

        if non_null_count > 0 and unique_count == non_null_count:
            print(f"  Potential unique key: {col}")


# ==========================================
# 6. ПОИСК КОЛОНОК С НИЗКОЙ ВАРИАТИВНОСТЬЮ
# ==========================================

print("\nLOW VARIABILITY COLUMNS")

for name, df in tables.items():

    low_variability = [
        col for col in df.columns
        if df[col].nunique(dropna=True) <= 1
    ]

    print(f"{name}: {low_variability}")

Количество загруженных таблиц: 8

Названия таблиц:
campaign_desc: (30, 4)
campaign_table: (7208, 3)
causal_data: (36786524, 5)
coupon: (124548, 3)
coupon_redempt: (2318, 4)
hh_demographic: (801, 8)
product: (92353, 7)
transaction_data: (2595732, 12)


,table,rows,columns,memory_mb,duplicates,missing_cells,missing_percentage
0,campaign_desc,30,4,0.00,0,0,0.0
1,campaign_table,7208,3,0.48,0,0,0.0
2,causal_data,36786524,5,4350.21,0,0,0.0
3,coupon,124548,3,2.85,5164,0,0.0
4,coupon_redempt,2318,4,0.07,0,0,0.0
5,hh_demographic,801,8,0.30,0,0,0.0
6,product,92353,7,27.60,0,0,0.0
7,transaction_data,2595732,12,237.65,0,0,0.0



TABLE: campaign_desc

1. FIRST 5 ROWS


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684



2. LAST 5 ROWS


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
25,TypeA,30,323,369
26,TypeB,29,281,334
27,TypeB,28,259,320
28,TypeC,27,237,300
29,TypeA,26,224,264


... (90 outputs left out)


7. DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
household_key,2595732.0,1.271953e+03,7.260660e+02,1.000000e+00,6.560000e+02,1.272000e+03,1.913000e+03,2.500000e+03
BASKET_ID,2595732.0,3.402620e+10,4.711649e+09,2.698485e+10,3.040805e+10,3.276081e+10,4.012685e+10,4.230536e+10
DAY,2595732.0,3.887562e+02,1.897210e+02,1.000000e+00,2.290000e+02,3.900000e+02,5.530000e+02,7.110000e+02
PRODUCT_ID,2595732.0,2.891435e+06,3.837404e+06,2.567100e+04,9.174590e+05,1.028816e+06,1.133018e+06,1.831630e+07
QUANTITY,2595732.0,1.004286e+02,1.153436e+03,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,8.963800e+04
SALES_VALUE,2595732.0,3.104120e+00,4.182274e+00,0.000000e+00,1.290000e+00,2.000000e+00,3.490000e+00,8.400000e+02
STORE_ID,2595732.0,3.142673e+03,8.937113e+03,1.000000e+00,3.300000e+02,3.720000e+02,4.220000e+02,3.428000e+04
RETAIL_DISC,2595732.0,-5.387054e-01,1.249191e+00,-1.800000e+02,-6.900000e-01,-1.000000e-02,0.000000e+00,3.990000e+00
TRANS_TIME,2595732.0,1.561586e+03,3.998378e+02,0.000000e+00,1.308000e+03,1.613000e+03,1.843000e+03,2.359000e+03
WEEK_NO,2595732.0,5.622150e+01,2.710223e+01,1.000000e+00,3.300000e+01,5.600000e+01,8.000000e+01,1.020000e+02


,table,column,dtype,non_null,missing,missing_pct,unique,unique_pct
0,campaign_desc,DESCRIPTION,str,30,0,0.0,3,10.00
1,campaign_desc,CAMPAIGN,int64,30,0,0.0,30,100.00
2,campaign_desc,START_DAY,int64,30,0,0.0,27,90.00
3,campaign_desc,END_DAY,int64,30,0,0.0,28,93.33
4,campaign_table,DESCRIPTION,str,7208,0,0.0,3,0.04
5,campaign_table,household_key,int64,7208,0,0.0,1584,21.98
6,campaign_table,CAMPAIGN,int64,7208,0,0.0,30,0.42
7,causal_data,PRODUCT_ID,int64,36786524,0,0.0,68377,0.19
8,causal_data,STORE_ID,int64,36786524,0,0.0,115,0.00
9,causal_data,WEEK_NO,int64,36786524,0,0.0,93,0.00



POTENTIAL KEY COLUMNS

campaign_desc
  Potential unique key: CAMPAIGN

campaign_table

causal_data

coupon

coupon_redempt

hh_demographic
  Potential unique key: household_key

product
  Potential unique key: PRODUCT_ID

transaction_data

LOW VARIABILITY COLUMNS
campaign_desc: []
campaign_table: []
causal_data: []
coupon: []
coupon_redempt: []
hh_demographic: []
product: []
transaction_data: []
